# AI Pest Finder — Backend

Cleaned final backend notebook for the AI Pest Finder application.

The backend uses FastAPI, the final MobileNetV2 pest classifier, and the relevance-gate model.

In [2]:
!pip install fastapi uvicorn python-multipart

## Backend configuration

The following paths point to the final trained models and the backend folder.

In [4]:
from pathlib import Path

backend_path = Path(r"C:\Users\jeeva\OneDrive\Desktop\AI pest finder\backend")
backend_path.mkdir(parents=True, exist_ok=True)

MODEL_PATH = r"C:\Users\jeeva\Downloads\ip102_v1.1\final_pest_mobilenetv2_74_46.keras"
MAPPING_PATH = r"C:\Users\jeeva\Downloads\ip102_v1.1\pest_class_mapping.json"
GATE_MODEL_PATH = r"C:\Users\jeeva\OneDrive\Desktop\AI pest finder\gate_model.keras"

print(backend_path)

C:\Users\jeeva\OneDrive\Desktop\AI pest finder\backend


## Create the final `main.py`

This replaces the older backend implementation. The class mapping is loaded directly from the JSON object using `mapping.items()`.

In [6]:
main_file = backend_path / "main.py"
main_code = 'from fastapi import FastAPI, File, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom PIL import Image\nimport tensorflow as tf\nimport numpy as np\nimport io\nimport json\n\nMODEL_PATH = r"C:\\Users\\jeeva\\Downloads\\ip102_v1.1\\final_pest_mobilenetv2_74_46.keras"\nMAPPING_PATH = r"C:\\Users\\jeeva\\Downloads\\ip102_v1.1\\pest_class_mapping.json"\nGATE_MODEL_PATH = r"C:\\Users\\jeeva\\OneDrive\\Desktop\\AI pest finder\\gate_model.keras"\n\nprint("Loading AI model...")\nmodel = tf.keras.models.load_model(MODEL_PATH)\nprint("AI model loaded successfully.")\n\nprint("Loading relevance gate model...")\ngate_model = tf.keras.models.load_model(GATE_MODEL_PATH)\nprint("Relevance gate model loaded successfully.")\n\nwith open(MAPPING_PATH, "r", encoding="utf-8") as f:\n    mapping = json.load(f)\n\nlabel_to_class = {\n    int(key): value\n    for key, value in mapping.items()\n}\n\nprint("Class mapping loaded successfully.")\n\napp = FastAPI(title="AI Pest Finder API")\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n@app.get("/")\ndef home():\n    return {\n        "status": "success",\n        "message": "AI Pest Finder API is running"\n    }\n\npest_info = {\n    "rice leaf roller": {\n        "management": "Remove heavily affected leaves, maintain field sanitation, and monitor the crop regularly."\n    },\n    "asiatic rice borer": {\n        "management": "Remove affected plant parts, monitor for infestation, and follow recommended rice pest management practices."\n    },\n    "brown plant hopper": {\n        "management": "Monitor hopper populations, avoid excessive nitrogen fertilizer, and use recommended integrated pest management practices."\n    },\n    "rice leafhopper": {\n        "management": "Monitor the crop regularly, remove heavily affected plants when appropriate, and follow recommended pest management practices."\n    },\n    "black cutworm": {\n        "management": "Remove weeds around the crop, monitor young plants, and follow recommended cutworm management practices."\n    },\n    "corn borer": {\n        "management": "Remove severely damaged plant material, monitor regularly, and use recommended integrated pest management methods."\n    },\n    "army worm": {\n        "management": "Inspect leaves regularly, remove heavily infested plant material, and follow recommended armyworm management practices."\n    },\n    "aphids": {\n        "management": "Remove heavily affected leaves, encourage natural predators, and use appropriate pest-control methods when necessary."\n    },\n    "Thrips": {\n        "management": "Remove heavily affected plant material, control weeds, monitor regularly, and follow recommended thrips management practices."\n    },\n    "Trialeurodes vaporariorum": {\n        "management": "Monitor the underside of leaves, remove heavily infested leaves, and use recommended whitefly management practices."\n    }\n}\n\n@app.post("/predict")\nasync def predict(file: UploadFile = File(...)):\n    if not file.content_type or not file.content_type.startswith("image/"):\n        return {\n            "success": False,\n            "error": "Please upload a valid image file."\n        }\n\n    try:\n        image_bytes = await file.read()\n        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")\n        image = image.resize((128, 128))\n\n        image_array = np.array(image, dtype=np.float32) / 255.0\n        image_batch = np.expand_dims(image_array, axis=0)\n\n        gate_probability = float(\n            gate_model.predict(image_batch, verbose=0)[0][0]\n        )\n\n        print("Gate probability:", round(gate_probability * 100, 2), "%")\n\n        if gate_probability < 0.50:\n            return {\n                "success": False,\n                "type": "not_supported",\n                "message": (\n                    "The uploaded image does not appear to contain a supported "\n                    "crop pest. Please upload a clear crop image showing one "\n                    "of the supported pests."\n                )\n            }\n\n        predictions = model.predict(image_batch, verbose=0)[0]\n        predicted_label = int(np.argmax(predictions))\n        confidence = float(predictions[predicted_label] * 100)\n        pest_name = label_to_class[predicted_label]\n\n        if confidence >= 70:\n            confidence_level = "High"\n        elif confidence >= 50:\n            confidence_level = "Moderate"\n        else:\n            confidence_level = "Low"\n\n        management = pest_info.get(\n            pest_name, {}\n        ).get(\n            "management",\n            "Follow recommended agricultural pest management practices."\n        )\n\n        return {\n            "success": True,\n            "pest": pest_name,\n            "confidence": round(confidence, 2),\n            "confidence_level": confidence_level,\n            "management": management\n        }\n\n    except Exception as e:\n        return {\n            "success": False,\n            "error": str(e)\n        }\n'
main_file.write_text(main_code, encoding="utf-8")
print("main.py created successfully.")
print(main_file)

main.py created successfully.
C:\Users\jeeva\OneDrive\Desktop\AI pest finder\backend\main.py


## Verify required files

This checks the final backend dependencies without starting duplicate servers or making repeated API requests.

In [8]:
required_files = {
    "Classifier model": Path(MODEL_PATH),
    "Class mapping": Path(MAPPING_PATH),
    "Relevance gate model": Path(GATE_MODEL_PATH),
    "Backend main.py": backend_path / "main.py"
}

for name, path in required_files.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

Classifier model: FOUND
Class mapping: FOUND
Relevance gate model: FOUND
Backend main.py: FOUND


## Start the backend

Run this in PowerShell from the backend folder:

```powershell
cd "C:\Users\jeeva\OneDrive\Desktop\AI pest finder\backend"
python -m uvicorn main:app --host 127.0.0.1 --port 8000
```

API: `http://127.0.0.1:8000`

Swagger documentation: `http://127.0.0.1:8000/docs`